In [1]:
pip install scipy

In [2]:
!pip install astropy

In [3]:
!pip install matplotlib

In [4]:
!pip install astroquery

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.9 MB/s eta 0:00:00


In [5]:
!pip install numpy

In [6]:
!pip install timeSeries

  Preparing metadata (setup.py) ... done
  Created wheel for timeSeries: filename=timeseries-0.5.0-py3-none-any.whl size=7197 sha256=f5660bdb6457336b8522a21395dfad05c6df363e9e4def454571771f14aec2ff
  Stored in directory: /root/.cache/pip/wheels/0f/68/13/8e208bae38a470475d987df54d59fbd02f5820536c17d9f880
Successfully built timeSeries


In [7]:
!pip install astropy requests tqdm

In [14]:
#!/usr/bin/env python3
"""
=============================================================================
COMPARACIÓN BAYESIANA DE MODELOS: UAT/UCP vs ΛCDM
con cronómetros cósmicos H(z) (31 puntos, Moresco et al. 2016,2020)

UAT: 1 parámetro libre (margen térmico 7%), ΛCDM: 6 parámetros libres
Métodos: AIC, BIC, factor de Bayes aproximado (Schwarz), escala de Jeffreys.
=============================================================================
"""

import numpy as np
from scipy.integrate import quad

# ======================================================================
# DATOS DE CRONÓMETROS CÓSMICOS (31 puntos)
# ======================================================================
# z, H(z) [km/s/Mpc], error [km/s/Mpc]
cosmic_chronometers = np.array([
    [0.07,  69.0,  19.6],
    [0.09,  69.0,  12.0],
    [0.12,  68.6,  26.2],
    [0.17,  83.0,  8.0],
    [0.179, 75.0,  4.0],
    [0.199, 75.0,  5.0],
    [0.20,  72.9,  29.6],
    [0.27,  77.0,  14.0],
    [0.28,  88.8,  36.6],
    [0.352, 83.0,  14.0],
    [0.38,  83.0,  13.5],
    [0.4,   95.0,  17.0],
    [0.4004,77.0, 10.2],
    [0.4247,87.1, 11.2],
    [0.4497,92.8, 12.9],
    [0.47,  89.0,  34.0],
    [0.4783,80.9, 9.0],
    [0.48,  97.0,  62.0],
    [0.593, 104.0, 13.0],
    [0.68,  92.0,  8.0],
    [0.781, 105.0, 12.0],
    [0.875, 125.0, 17.0],
    [0.88,  90.0,  40.0],
    [0.9,   117.0, 23.0],
    [1.037, 154.0, 20.0],
    [1.3,   168.0, 17.0],
    [1.363, 160.0, 33.6],
    [1.43,  177.0, 18.0],
    [1.53,  140.0, 14.0],
    [1.75,  202.0, 40.0],
    [1.965, 186.5, 50.4],
])

z_cc = cosmic_chronometers[:, 0]
H_obs = cosmic_chronometers[:, 1]
H_err = cosmic_chronometers[:, 2]

# ======================================================================
# PARÁMETROS FIJADOS DE LOS MODELOS
# ======================================================================

# UAT/UCP (actualizado)
k_early = 0.967
H0_uat = 73.04               # km/s/Mpc (SH0ES)
Omega_m_uat = 0.3133         # Ω_m^(eff)
Omega_L_uat = 0.6867
Omega_r = 7.79e-5            # radiación

# ΛCDM (Planck 2018)
H0_lcdm = 67.36
Omega_m_lcdm = 0.315
Omega_L_lcdm = 0.685

# ======================================================================
# FUNCIONES DE EXPANSIÓN H(z)
# ======================================================================
def H_uat(z):
    return H0_uat * np.sqrt(
        k_early * Omega_r * (1+z)**4 +
        k_early * Omega_m_uat * (1+z)**3 +
        Omega_L_uat
    )

def H_lcdm(z):
    return H0_lcdm * np.sqrt(
        Omega_r * (1+z)**4 +
        Omega_m_lcdm * (1+z)**3 +
        Omega_L_lcdm
    )

# ======================================================================
# CÁLCULO DE χ²
# ======================================================================
H_uat_pred = H_uat(z_cc)
H_lcdm_pred = H_lcdm(z_cc)

chi2_uat = np.sum(((H_obs - H_uat_pred) / H_err)**2)
chi2_lcdm = np.sum(((H_obs - H_lcdm_pred) / H_err)**2)

print(f"χ² UAT   = {chi2_uat:.2f}")
print(f"χ² ΛCDM  = {chi2_lcdm:.2f}")

# ======================================================================
# CRITERIOS DE INFORMACIÓN
# ======================================================================
N = len(z_cc)                # número de datos
k_uat = 1                    # solo el margen del 7%
k_lcdm = 6                   # H0, Ωm, ΩΛ, σ8, n_s, τ

# AIC
aic_uat = chi2_uat + 2 * k_uat
aic_lcdm = chi2_lcdm + 2 * k_lcdm
delta_aic = aic_uat - aic_lcdm

# BIC
log_N = np.log(N)
bic_uat = chi2_uat + k_uat * log_N
bic_lcdm = chi2_lcdm + k_lcdm * log_N
delta_bic = bic_uat - bic_lcdm

# Factor de Bayes aproximado (Schwarz)
log_B = -0.5 * (chi2_uat - chi2_lcdm) + 0.5 * (k_lcdm - k_uat) * log_N
B = np.exp(log_B)

# Escala de Jeffreys
if B > 100:
    evidence = "Decisiva para UAT (B > 100)"
elif B > 10:
    evidence = "Fuerte para UAT (10 < B ≤ 100)"
elif B > 3:
    evidence = "Sustancial para UAT (3 < B ≤ 10)"
elif B > 1:
    evidence = "Débil para UAT (1 < B ≤ 3)"
elif B > 1/3:
    evidence = "Inconcluyente (1/3 < B < 1)"
elif B > 1/10:
    evidence = "Sustancial para ΛCDM (1/10 < B ≤ 1/3)"
elif B > 1/100:
    evidence = "Fuerte para ΛCDM (1/100 < B ≤ 1/10)"
else:
    evidence = "Decisiva para ΛCDM (B ≤ 1/100)"

# ======================================================================
# RESULTADOS
# ======================================================================
print(f"\n--- Model Selection (N={N}) ---")
print(f"UAT:   k={k_uat}, χ²/dof={chi2_uat/(N-k_uat):.4f}")
print(f"ΛCDM:  k={k_lcdm}, χ²/dof={chi2_lcdm/(N-k_lcdm):.4f}")
print(f"\nΔAIC = {delta_aic:+.2f}")
print(f"ΔBIC = {delta_bic:+.2f}")
print(f"log(B) = {log_B:.4f}")
print(f"Factor de Bayes B = {B:.2f}")
print(f"Evidencia (Jeffreys): {evidence}")

χ² UAT   = 20.71
χ² ΛCDM  = 14.90

--- Model Selection (N=31) ---
UAT:   k=1, χ²/dof=0.6903
ΛCDM:  k=6, χ²/dof=0.5960

ΔAIC = -4.19
ΔBIC = -11.36
log(B) = 5.6815
Factor de Bayes B = 293.38
Evidencia (Jeffreys): Decisiva para UAT (B > 100)
